In [ ]:
import pandas as pd
df = pd.read_excel("/content/sentry_threat_language_dataset_2000.xlsx")
df.head()

,text,label,category
0,I am debugging my code today,0,SAFE
1,Someone is trying to break in immediately,1,THREAT
2,I am meeting my friend this afternoon,0,SAFE
3,Move away from us and I need help,1,THREAT
4,I am cooking dinner with my friends,0,SAFE


In [ ]:
print("Dataset size:", len(df))

print("Columns:", df.columns.tolist())

print(df["label"].value_counts())

print(df.isnull().sum())

Dataset size: 2000
Columns: ['text', 'label', 'category']
label
0    1000
1    1000
Name: count, dtype: int64
text        0
label       0
category    0
dtype: int64


In [ ]:
from sklearn.model_selection import train_test_split
X = df['text']
y = df['label']
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)
print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Training: 1600
Validation: 200
Test: 200


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
text = X_train.iloc[0]

encoded = tokenizer(
    text,
    padding="max_length",
    truncation=True,
    max_length=64,
    return_tensors="pt"
)

print("Text:")
print(text)

print("\nInput IDs:")
print(encoded["input_ids"])

print("\nAttention Mask:")
print(encoded["attention_mask"])

Text:
Stop following us right now

Input IDs:
tensor([[ 101, 2644, 2206, 2149, 2157, 2085,  102,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0]])

Attention Mask:
tensor([[1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])


In [ ]:
train_encodings = tokenizer(
    X_train.tolist(),
    padding = "max_length",
    truncation = True,
    max_length = 64
)

In [ ]:
val_encodings = tokenizer(
    X_val.tolist(),
    padding="max_length",
    truncation=True,
    max_length=64
)

test_encodings = tokenizer(
    X_test.tolist(),
    padding="max_length",
    truncation=True,
    max_length=64
)


In [ ]:
print("Training examples:", len(train_encodings["input_ids"]))
print("Validation examples:", len(val_encodings["input_ids"]))
print("Test examples:", len(test_encodings["input_ids"]))

Training examples: 1600
Validation examples: 200
Test examples: 200


In [ ]:
from torch.utils.data import Dataset, DataLoader

class SentryDataset(Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(value[idx])
            for key, value in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item


In [ ]:
train_dataset = SentryDataset(
    train_encodings,
    y_train
)

val_dataset = SentryDataset(
    val_encodings,
    y_val
)

test_dataset = SentryDataset(
    test_encodings,
    y_test
)

print("Train dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Test dataset:", len(test_dataset))

Train dataset: 1600
Validation dataset: 200
Test dataset: 200


In [ ]:
import torch
import pandas as pd
from torch.utils.data import Dataset,DataLoader
train_loader = DataLoader( train_dataset, batch_size = 16 , shuffle = True)
val_loader = DataLoader( val_dataset,
                        batch_size = 16,
                         shuffle = False)
test_loader = DataLoader( test_dataset,batch_size = 16,shuffle = False)

In [ ]:
batch = next(iter(train_loader))
print("One training batch:")
print("Input IDs shape:",
      batch["input_ids"].shape)

print("Attention mask shape:",
      batch["attention_mask"].shape)

print("Labels shape:",
      batch["labels"].shape)

One training batch:
Input IDs shape: torch.Size([16, 64])
Attention mask shape: torch.Size([16, 64])
Labels shape: torch.Size([16])


In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nUsing device:", device)


Using device: cuda


In [ ]:
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
model =  AutoModel.from_pretrained("distilbert-base-uncased")


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
for param in model.parameters():
  param.requires_grad = False


In [ ]:
import torch.nn as nn
classifier = nn.Linear(768,2)

In [ ]:
model = model.to(device)

classifier = classifier.to(device)

In [ ]:
loss_function = nn.CrossEntropyLoss()


In [ ]:
optimizer = torch.optim.Adam(classifier.parameters(),lr = 0.001)

In [ ]:
num_epochs = 5
for epoch in range(num_epochs):
  classifier.train()
  total_loss = 0
  for batch in train_loader:
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    labels = batch["labels"].to(device)

    with torch.no_grad():
      output = model(input_ids = input_ids,
                     attention_mask = attention_mask)
    cls_vector = output.last_hidden_state[:, 0, :]
    logits = classifier(cls_vector)
    loss = loss_function(logits,labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss += loss.item()


In [ ]:


num_epochs = 5

for epoch in range(num_epochs):

    classifier.train()

    total_train_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.no_grad():

            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )


        cls_vector = output.last_hidden_state[:, 0, :]


        logits = classifier(cls_vector)

        loss = loss_function(logits, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item()

    classifier.eval()

    total_val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            output = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            cls_vector = output.last_hidden_state[:, 0, :]

            logits = classifier(cls_vector)

            loss = loss_function(logits, labels)

            total_val_loss += loss.item()

            predictions = torch.argmax(logits, dim=1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)


    val_accuracy = correct / total




    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {total_train_loss:.4f} | "
        f"Val Loss: {total_val_loss:.4f} | "
        f"Val Accuracy: {val_accuracy:.4f}"
    )

Epoch 1/5 | Train Loss: 1.8297 | Val Loss: 0.2509 | Val Accuracy: 1.0000
Epoch 2/5 | Train Loss: 1.6337 | Val Loss: 0.2296 | Val Accuracy: 1.0000
Epoch 3/5 | Train Loss: 1.5053 | Val Loss: 0.2139 | Val Accuracy: 1.0000
Epoch 4/5 | Train Loss: 1.3850 | Val Loss: 0.1981 | Val Accuracy: 1.0000
Epoch 5/5 | Train Loss: 1.2482 | Val Loss: 0.1825 | Val Accuracy: 1.0000


In [ ]:


classifier.eval()

correct = 0
total = 0

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        output = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_vector = output.last_hidden_state[:, 0, :]

        logits = classifier(cls_vector)

        predictions = torch.argmax(logits, dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total

print(f"Test Accuracy: {test_accuracy:.4f}")

Test Accuracy: 1.0000


In [ ]:


def predict(text):

    encoded = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=64
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }


    with torch.no_grad():

        output = model(**encoded)


        cls_vector = output.last_hidden_state[:, 0, :]

        logits = classifier(cls_vector)

        probabilities = torch.softmax(logits, dim=1)

        prediction = torch.argmax(probabilities, dim=1)

    return probabilities, prediction

In [ ]:
test_sentences = [
    "The movie showed someone being murdered.",
    "I read about an attack in the newspaper.",
    "I'm going home after work.",
    "My friend threatened to beat me in a video game.",

    "Don't come any closer.",
    "I'm going to hurt you.",
    "If you leave, I'll find you.",
    "Give me my phone or you're going to regret it.",
    "I know where you live."
]

for text in test_sentences:

    probabilities, prediction = predict(text)

    predicted_class = prediction.item()

    if predicted_class == 0:
        label = "SAFE"
    else:
        label = "THREAT"

    confidence = probabilities[0][predicted_class].item()

    print(f"Text: {text}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.4f}")
    print("-" * 60)

Text: The movie showed someone being murdered.
Prediction: SAFE
Confidence: 0.7833
------------------------------------------------------------
Text: I read about an attack in the newspaper.
Prediction: SAFE
Confidence: 0.6146
------------------------------------------------------------
Text: I'm going home after work.
Prediction: SAFE
Confidence: 0.9913
------------------------------------------------------------
Text: My friend threatened to beat me in a video game.
Prediction: THREAT
Confidence: 0.6090
------------------------------------------------------------
Text: Don't come any closer.
Prediction: THREAT
Confidence: 0.9999
------------------------------------------------------------
Text: I'm going to hurt you.
Prediction: THREAT
Confidence: 0.9984
------------------------------------------------------------
Text: If you leave, I'll find you.
Prediction: THREAT
Confidence: 0.9714
------------------------------------------------------------
Text: Give me my phone or you're going

In [ ]:
import os
import json
import torch
import shutil

export_dir = "/content/sentry_threat_detector"
os.makedirs(export_dir, exist_ok=True)

torch.save(
    classifier.state_dict(),
    f"{export_dir}/sentry_threat_classifier.pth"
)

tokenizer.save_pretrained(
    f"{export_dir}/tokenizer"
)

config = {
    "base_model": "distilbert-base-uncased",
    "task": "threat_language_detection",

    "labels": {
        "0": "SAFE",
        "1": "THREAT"
    },

    "architecture": {
        "transformer": "DistilBERT",
        "hidden_size": 768,
        "classifier": "Linear(768, 2)"
    },

    "max_length": 64,

    "distilbert_frozen": True,

    "description": (
        "Sentry threat language detector. "
        "DistilBERT generates a 768-dimensional CLS representation, "
        "which is passed through a linear classifier to predict SAFE or THREAT."
    )
}

with open(
    f"{export_dir}/config.json",
    "w"
) as f:
    json.dump(config, f, indent=4)

inference_code = '''
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel


class SentryThreatDetector:

    def __init__(self, model_path, tokenizer_path):

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            tokenizer_path
        )

        # Load pretrained DistilBERT
        self.model = AutoModel.from_pretrained(
            "distilbert-base-uncased"
        )

        # Classifier
        self.classifier = nn.Linear(768, 2)

        # Load trained classifier weights
        self.classifier.load_state_dict(
            torch.load(
                model_path,
                map_location=self.device
            )
        )

        self.model.to(self.device)
        self.classifier.to(self.device)

        self.model.eval()
        self.classifier.eval()


    def predict(self, text):

        encoded = self.tokenizer(
            text,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=64
        )

        encoded = {
            key: value.to(self.device)
            for key, value in encoded.items()
        }

        with torch.no_grad():

            output = self.model(**encoded)

            # First token = CLS representation
            cls_vector = output.last_hidden_state[:, 0, :]

            # Classifier
            logits = self.classifier(cls_vector)

            # Probabilities
            probabilities = torch.softmax(
                logits,
                dim=1
            )

            prediction = torch.argmax(
                probabilities,
                dim=1
            )

        predicted_class = prediction.item()

        if predicted_class == 0:
            label = "SAFE"
        else:
            label = "THREAT"

        confidence = probabilities[
            0, predicted_class
        ].item()

        return {
            "label": label,
            "confidence": confidence,
            "threat_probability": probabilities[0, 1].item(),
            "safe_probability": probabilities[0, 0].item()
        }
'''

with open(
    f"{export_dir}/inference.py",
    "w"
) as f:
    f.write(inference_code)

readme = '''
# Sentry Threat Language Detector

## Model

Base model:
distilbert-base-uncased

Task:
Threat language detection

## Classes

0 = SAFE
1 = THREAT

## Architecture

Text
↓
DistilBERT tokenizer
↓
DistilBERT
↓
CLS representation (768 dimensions)
↓
Linear classifier (768 → 2)
↓
SAFE / THREAT

## Training

DistilBERT was frozen during training.

Only the Linear(768, 2) classifier was trained.

Maximum sequence length:
64

## Files

sentry_threat_classifier.pth
    Trained classifier weights.

tokenizer/
    DistilBERT tokenizer files.

config.json
    Model configuration.

inference.py
    Python inference class.

## Example

from inference import SentryThreatDetector

detector = SentryThreatDetector(
    "sentry_threat_classifier.pth",
    "tokenizer"
)

result = detector.predict(
    "I'm going to hurt you"
)

print(result)

Expected output:

{
    "label": "THREAT",
    "confidence": ...,
    "threat_probability": ...,
    "safe_probability": ...
}
'''

with open(
    f"{export_dir}/README.md",
    "w"
) as f:
    f.write(readme)


zip_path = shutil.make_archive(
    "/content/sentry_threat_detector",
    "zip",
    export_dir
)

print("==========================================")
print("SENTRY MODEL EXPORT COMPLETE")
print("==========================================")
print()
print("Files exported:")
print("✓ sentry_threat_classifier.pth")
print("✓ tokenizer/")
print("✓ config.json")
print("✓ inference.py")
print("✓ README.md")
print()
print(f"ZIP file: {zip_path}")

SENTRY MODEL EXPORT COMPLETE

Files exported:
✓ sentry_threat_classifier.pth
✓ tokenizer/
✓ config.json
✓ inference.py
✓ README.md

ZIP file: /content/sentry_threat_detector.zip
